In [1]:
import streamlit as st
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn.model_selection import cross_val_score,train_test_split
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor,AdaBoostRegressor
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.neighbors import KNeighborsRegressor
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')

In [2]:
data=pd.read_csv('https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv')
data.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [3]:
Sex_encoder= LabelEncoder()
Smoker_encoder= LabelEncoder()
Region_encoder= LabelEncoder()

In [4]:
data["Sex"] = Sex_encoder.fit_transform(data.sex)
data["Smoker"] = Smoker_encoder.fit_transform(data.smoker)
data["Region"] = Region_encoder.fit_transform(data.region)
data.head()

,age,sex,bmi,children,smoker,region,charges,Sex,Smoker,Region
0,19,female,27.900,0,yes,southwest,16884.92400,0,1,3
1,18,male,33.770,1,no,southeast,1725.55230,1,0,2
2,28,male,33.000,3,no,southeast,4449.46200,1,0,2
3,33,male,22.705,0,no,northwest,21984.47061,1,0,1
4,32,male,28.880,0,no,northwest,3866.85520,1,0,1


In [5]:
X= data[['age', 'Sex', 'bmi', 'children',  'Smoker','Region']].values
X

array([[19.  ,  0.  , 27.9 ,  0.  ,  1.  ,  3.  ],
       [18.  ,  1.  , 33.77,  1.  ,  0.  ,  2.  ],
       [28.  ,  1.  , 33.  ,  3.  ,  0.  ,  2.  ],
       ...,
       [18.  ,  0.  , 36.85,  0.  ,  0.  ,  2.  ],
       [21.  ,  0.  , 25.8 ,  0.  ,  0.  ,  3.  ],
       [61.  ,  0.  , 29.07,  0.  ,  1.  ,  1.  ]], shape=(1338, 6))

In [6]:
Y = data[["charges"]].values
Y

array([[16884.924 ],
       [ 1725.5523],
       [ 4449.462 ],
       ...,
       [ 1629.8335],
       [ 2007.945 ],
       [29141.3603]], shape=(1338, 1))

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [8]:
model = Sequential()

In [9]:
model.add(Input(shape =  (X_train.shape[1],)))
model.add(Dense(15, activation = 'relu'))
model.add(Dropout(0.2))
model.add(Dense(120, activation = 'relu'))
model.add(Dropout(0.2))
model.add(Dense(100, activation = 'relu'))
model.add(Dropout(0.2))
model.add(Dense(80, activation = 'relu'))
model.add(Dense(1, activation = 'relu'))

In [10]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 15)             │           105 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 15)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 120)            │         1,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 120)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 100)            │        12,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 80)             │         8,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            81 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,286 (87.05 KB)

 Trainable params: 22,286 (87.05 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
model.compile(optimizer= 'adam', loss= "mean_squared_error")

In [12]:
X_train.shape

(1070, 6)

In [13]:
model.fit(X_train, y_train, batch_size=10, epochs=10, validation_data= (X_test, y_test), callbacks=EarlyStopping(patience=5))

Epoch 1/10
107/107 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 258101408.0000 - val_loss: 141873968.0000
Epoch 2/10
107/107 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 138128688.0000 - val_loss: 132213560.0000
Epoch 3/10
107/107 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 137268176.0000 - val_loss: 132135680.0000
Epoch 4/10
107/107 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 135771808.0000 - val_loss: 131301608.0000
Epoch 5/10
107/107 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 134038960.0000 - val_loss: 129864096.0000
Epoch 6/10
107/107 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 134605376.0000 - val_loss: 130361768.0000
Epoch 7/10
107/107 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 133087816.0000 - val_loss: 129805128.0000
Epoch 8/10
107/107 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 133481152.0000 - val_loss: 128196256.0000
Epoch 9/10
107/107 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 131549184.0000 - val_loss: 127149176.0000
Epoch 10/10
107/107 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 128467752.0000 - val_

In [14]:
pred = model.predict(X_test)
pred

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step


array([[14674.362 ],
       [13074.907 ],
       [19626.121 ],
       [15243.185 ],
       [11117.187 ],
       [15170.365 ],
       [ 8174.3394],
       [20618.744 ],
       [ 9241.69  ],
       [16397.992 ],
       [12397.202 ],
       [14811.462 ],
       [11634.99  ],
       [19115.9   ],
       [21204.467 ],
       [18500.762 ],
       [18382.11  ],
       [17709.297 ],
       [14935.6455],
       [15057.808 ],
       [10630.776 ],
       [15351.609 ],
       [ 9576.792 ],
       [12124.257 ],
       [16339.96  ],
       [17512.191 ],
       [18848.783 ],
       [11892.36  ],
       [15351.749 ],
       [ 9438.496 ],
       [14368.626 ],
       [17886.3   ],
       [10771.194 ],
       [10618.175 ],
       [11141.795 ],
       [17021.893 ],
       [ 9282.3545],
       [14496.702 ],
       [16739.617 ],
       [15326.254 ],
       [10097.951 ],
       [10790.999 ],
       [18309.836 ],
       [17425.465 ],
       [13963.798 ],
       [17712.145 ],
       [11054.302 ],
       [10130

In [15]:
history = model.history

In [16]:
history.history

{'loss': [258101408.0,
  138128688.0,
  137268176.0,
  135771808.0,
  134038960.0,
  134605376.0,
  133087816.0,
  133481152.0,
  131549184.0,
  128467752.0],
 'val_loss': [141873968.0,
  132213560.0,
  132135680.0,
  131301608.0,
  129864096.0,
  130361768.0,
  129805128.0,
  128196256.0,
  127149176.0,
  126388240.0]}

In [17]:
plt.plot(history.history["loss"])
plt.plot(history.history["val_loss"])
plt.show()